In [1]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e .[torch,metrics]


Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 672, done.
remote: Counting objects: 100% (672/672), done.
remote: Compressing objects: 100% (510/510), done.
remote: Total 672 (delta 156), reused 429 (delta 100), pack-reused 0 (from 0)
Receiving objects: 100% (672/672), 5.35 MiB | 22.94 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/content/LLaMA-Factory
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
!llamafactory-cli train colab_llamafactory_config.yaml


/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-06-06 07:17:45] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
config.json: 100% 660/660 [00:00<00:00, 3.57MB/s]
[INFO|configuration_utils.py:771] 2026-06-06 07:17:45,595 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instru

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "saves/qwen2.5-1.5b/lora/sft"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [7]:
prompt = """
You are a structured invoice data extraction assistant. Extract invoice data from the input text and return ONLY valid JSON matching the schema.
plz issue bill for Vertex Marketting, emal vertex@mail.com, addres 39 main road. 3 Photography Package AED 6334, 1 Monitor Arm AED 3986, 1 Conference Table AED 2408. disc 0% and tax 5%. due 2026-09-12. note soft copy.
"""

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

You are a structured invoice data extraction assistant. Extract invoice data from the input text and return ONLY valid JSON matching the schema.
plz issue bill for Vertex Marketting, emal vertex@mail.com, addres 39 main road. 3 Photography Package AED 6334, 1 Monitor Arm AED 3986, 1 Conference Table AED 2408. disc 0% and tax 5%. due 2026-09-12. note soft copy.

assistant
{"customerInfo":{"name":"Vertex Marketing","email":"vertex@mail.com","address":"39 main road"},"items":[{"description":"Photography Package","quantity":3,"rate":6334},{"description":"Monitor Arm","quantity":1,"rate":3986},{"description":"Conference Table","quantity":1,"rate":2408}],"taxRate":5,"notes":"soft copy","currency":"AED","paymentTerms":"net-30","dueDate":"2026-09-12","discountRate":0,"templateId":"minimal-executive","title":"Vertex Marketing Invoice","themeColor":"#7C3AED"}


In [8]:
!llamafactory-cli export \
    --model_name_or_path Qwen/Qwen2.5-1.5B-Instruct \
    --adapter_name_or_path saves/qwen2.5-1.5b/lora/sft \
    --template qwen \
    --finetuning_type lora \
    --export_dir invoice-qwen2.5-1.5b-merged \
    --export_size 2 \
    --export_device cpu


[INFO|configuration_utils.py:771] 2026-06-06 08:07:38,605 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:847] 2026-06-06 08:07:38,613 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

merged_model_path = "invoice-qwen2.5-1.5b-merged"

tokenizer = AutoTokenizer.from_pretrained(merged_model_path)
model = AutoModelForCausalLM.from_pretrained(
    merged_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model.eval()

prompt = """You are a structured invoice data extraction assistant.
Extract invoice data from the input text and return ONLY valid JSON matching the schema.

plz issue bill for Vertex Marketting, emal vertex@mail.com, addres 39 main road.
3 Photography Package AED 6334, 1 Monitor Arm AED 3986, 1 Conference Table AED 2408.
disc 0% and tax 5%. due 2026-09-12. note soft copy.
"""

messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
You are a structured invoice data extraction assistant.
Extract invoice data from the input text and return ONLY valid JSON matching the schema.

plz issue bill for Vertex Marketting, emal vertex@mail.com, addres 39 main road.
3 Photography Package AED 6334, 1 Monitor Arm AED 3986, 1 Conference Table AED 2408.
disc 0% and tax 5%. due 2026-09-12. note soft copy.

assistant
{"customerInfo":{"name":"Vertex Marketing","email":"vertex@mail.com","address":"39 main road"},"items":[{"description":"Photography Package","quantity":3,"rate":6334},{"description":"Monitor Arm","quantity":1,"rate":3986},{"description":"Conference Table","quantity":1,"rate":2408}],"taxRate":5,"notes":"soft copy","currency":"AED","paymentTerms":"net-30","dueDate":"2026-09-12","discountRate":0,"templateId":"minimal-executive","title":"Vertex Marketing Invoice","themeColor":"#7C3AED"}


In [13]:
from transformers import AutoTokenizer

# Download the pristine tokenizer from the original model
clean_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

# Overwrite the corrupted tokenizer files in our merged directory
clean_tokenizer.save_pretrained("/content/LLaMA-Factory/invoice-qwen2.5-1.5b-merged")
print("Tokenizer successfully fixed!")


Tokenizer successfully fixed!


In [14]:
%cd /content
!git clone https://github.com/ggerganov/llama.cpp
!pip install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py LLaMA-Factory/invoice-qwen2.5-1.5b-merged \
    --outfile invoice-qwen2.5-1.5b-q8_0.gguf \
    --outtype q8_0


/content
fatal: destination path 'llama.cpp' already exists and is not an empty directory.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
INFO:hf-to-gguf:Loading model: invoice-qwen2.5-1.5b-merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:h

In [18]:
%cd /content
!python llama.cpp/convert_hf_to_gguf.py LLaMA-Factory/invoice-qwen2.5-1.5b-merged \
    --outfile /content/invoice-qwen2.5-1.5b-q8_0.gguf \
    --outtype q8_0


/content
INFO:hf-to-gguf:Loading model: invoice-qwen2.5-1.5b-merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> Q8_0, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> Q8_0, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:

In [19]:
import json, pathlib

cfg_path = pathlib.Path("LLaMA-Factory/invoice-qwen2.5-1.5b-merged/tokenizer_config.json")
cfg = json.loads(cfg_path.read_text())

print("Before:", type(cfg.get("extra_special_tokens")), "→", cfg.get("extra_special_tokens"))

# Fix: if it's a list, convert to an empty dict (the tokens are already
# registered as added_tokens_decoder — this field is safe to clear)
if isinstance(cfg.get("extra_special_tokens"), list):
    cfg["extra_special_tokens"] = {}
    cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
    print("✅ Patched tokenizer_config.json — extra_special_tokens is now {}")
else:
    print("No patch needed.")

Before: <class 'list'> → ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']
✅ Patched tokenizer_config.json — extra_special_tokens is now {}


In [20]:
%cd /content
!python llama.cpp/convert_hf_to_gguf.py \
    LLaMA-Factory/invoice-qwen2.5-1.5b-merged \
    --outfile /content/invoice-qwen2.5-1.5b-q8_0.gguf \
    --outtype q8_0

/content
INFO:hf-to-gguf:Loading model: invoice-qwen2.5-1.5b-merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> Q8_0, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> Q8_0, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:

In [1]:
from google.colab import files
files.download('/content/invoice-qwen2.5-1.5b-q8_0.gguf')


FileNotFoundError: Cannot find file: /content/invoice-qwen2.5-1.5b-q8_0.gguf